# GDP Prediction — ML (Polynomial, XGBoost) & DL (PyTorch LSTM + Transformer)

This Colab notebook trains Polynomial Regression and XGBoost (ML) and LSTM + Transformer encoder (PyTorch) for GDP prediction.

**Key points**
- Uses the CSV `final GDP Dataset - Copy.csv`. If not present, upload it when prompted.
- Prints DL training progress each epoch in the format: `Epoch i/N Train loss: <value> | Test loss: <value>`.
- Evaluation metrics used: **MAE**, **RMSE**, **R²** only.
- Produces comparison table and plots.

In [ ]:
# Install required packages (run in Colab; if packages already installed this is fast)
!pip install -q xgboost joblib

# Imports and device setup
import os
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
import xgboost as xgb

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

## 2) Metrics utilities
We will compute MAE, RMSE, and R² only (no accuracy).

In [ ]:
# Metrics utilities: MAE, RMSE, R2
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def r2(y_true, y_pred):
    return r2_score(y_true, y_pred)

def evaluate_metrics(y_true, y_pred):
    return {
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "R2": r2(y_true, y_pred)
    }

## 3) Load Data
- The script tries to read `/mnt/data/final GDP Dataset - Copy.csv`.  
- If not present, you'll be prompted to upload the CSV file from your computer (Colab file picker).
- After loading, we show a plot: Year vs GDP to visualize the target time series.

In [ ]:
from pathlib import Path

# Expected path (works if you uploaded file to Colab root or mounted drive)
expected_path = '/mnt/data/final GDP Dataset - Copy.csv'
local_path = 'final GDP Dataset - Copy.csv'

# Use whichever exists or ask user to upload
csv_path = None
if Path(expected_path).exists():
    csv_path = expected_path
elif Path(local_path).exists():
    csv_path = local_path
else:
    # Colab upload
    try:
        from google.colab import files
        print("Please upload the CSV file (choose 'final GDP Dataset - Copy.csv') — a file dialog will open.")
        uploaded = files.upload()
        # take first uploaded file
        first_key = next(iter(uploaded.keys()))
        csv_path = first_key
    except Exception as e:
        raise RuntimeError("CSV not found. Please upload `final GDP Dataset - Copy.csv` to the Colab environment.") from e

print("Using CSV file:", csv_path)

# Read CSV
df = pd.read_csv(csv_path)

# Normalize Year column to integer (extract first 4-digit year)
df['Year'] = df['Year'].astype(str).str.extract(r'(\d{4})').astype(int)

# Show head & info
print(df.head())
print(df.info())

# Plot Year vs GDP (target)
target_col = 'Gross Domestic Product__Constant Prices'
if target_col not in df.columns:
    raise RuntimeError(f"Expected target column '{target_col}' not found in CSV. Columns: {df.columns.tolist()}")

plt.figure(figsize=(10,4))
plt.plot(df['Year'], df[target_col], marker='o')
plt.title('Year vs Gross Domestic Product (Constant Prices)')
plt.xlabel('Year')
plt.ylabel('GDP (Constant Prices)')
plt.grid(True)
plt.tight_layout()
plt.show()

## 4) ML part: train/test 80/20 (time ordered)
This section:
- Splits the data (first 80% years for training, last 20% for testing).
- Standard-scales ML features.
- Trains Polynomial Regression (degree=2 + Ridge) and XGBoost.
- Evaluates and prints MAE, RMSE, R² for both models.

In [ ]:
# Prepare features and target
feature_cols = [c for c in df.columns if c not in ['Year', target_col]]
print("Feature columns:", feature_cols)

X = df[feature_cols].values
y = df[target_col].values

# Time-aware 80/20 split
n = len(df)
split_idx = int(n * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
years_ml = df['Year'].iloc[split_idx:].values

print(f"Total rows: {n}, Train rows: {len(y_train)}, Test rows: {len(y_test)}")

### 4.1 Standard scaling for ML features
We scale features before training ML models.

In [ ]:
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)
X_test_scaled = scaler_ml.transform(X_test)
# Save scaler for later
joblib.dump(scaler_ml, 'scaler_ml.joblib')

### 4.2 Polynomial Regression (degree=2) + Ridge

In [ ]:
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0, random_state=SEED))
])

poly_pipeline.fit(X_train, y_train)
poly_pred = poly_pipeline.predict(X_test)

metrics_poly = evaluate_metrics(y_test, poly_pred)
print("Polynomial Regression metrics:")
for k,v in metrics_poly.items():
    print(f"{k}: {v:.6f}")
# save
joblib.dump(poly_pipeline, 'polynomial_pipeline.joblib')

### 4.3 XGBoost (baseline hyperparameters)

In [ ]:
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=SEED,
                             n_estimators=200, learning_rate=0.05, max_depth=4, n_jobs=-1)
xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)

metrics_xgb = evaluate_metrics(y_test, xgb_pred)
print("XGBoost metrics:")
for k,v in metrics_xgb.items():
    print(f"{k}: {v:.6f}")
# save
joblib.dump(xgb_model, 'xgboost_model.joblib')

## 5) DL prep
We will:
- Fit a feature scaler (MinMax) on features for DL inputs.
- Fit a target-only scaler (MinMax) on the GDP target for safe inverse transforms.

In [ ]:
FEATURE_SCALER = MinMaxScaler()
TARGET_SCALER = MinMaxScaler()

features_all = df[feature_cols].values.astype(np.float32)
target_all = df[target_col].values.reshape(-1,1).astype(np.float32)

features_scaled_all = FEATURE_SCALER.fit_transform(features_all)
target_scaled_all = TARGET_SCALER.fit_transform(target_all)

joblib.dump(FEATURE_SCALER, 'scaler_features_dl.joblib')
joblib.dump(TARGET_SCALER, 'scaler_target_dl.joblib')

### 5.1 Create sequences (sliding window)
We create sequences of length `LOOKBACK` (default 5). Each sample's input is previous `LOOKBACK` rows of features and target is the GDP of the next year.

In [ ]:
LOOKBACK = 5  # you can change this value

X_seq = []
y_seq = []
for i in range(LOOKBACK, len(df)):
    X_seq.append(features_scaled_all[i-LOOKBACK:i, :])  # shape: (LOOKBACK, n_features)
    y_seq.append(target_scaled_all[i, 0])

X_seq = np.array(X_seq)  # (samples, LOOKBACK, features)
y_seq = np.array(y_seq)  # (samples,)

print("Sequence shapes:", X_seq.shape, y_seq.shape)

# Train/test split for DL sequences (time-ordered 80/20)
n_seq = X_seq.shape[0]
split_seq = int(n_seq * 0.8)
X_train_dl, X_test_dl = X_seq[:split_seq], X_seq[split_seq:]
y_train_dl, y_test_dl = y_seq[:split_seq], y_seq[split_seq:]
years_dl = df['Year'].iloc[LOOKBACK + split_seq : LOOKBACK + split_seq + len(y_test_dl)].values

print(f"DL train samples: {len(y_train_dl)}, DL test samples: {len(y_test_dl)}")
print("Years ML test length:", len(years_ml), "Years DL test length:", len(years_dl))

### 5.2 PyTorch Dataset & DataLoader
Create a small Dataset wrapper and DataLoader for training.

In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 8
train_dataset = SeqDataset(X_train_dl, y_train_dl)
test_dataset = SeqDataset(X_test_dl, y_test_dl)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 5.3 LSTM Model (PyTorch)
Bidirectional LSTM -> Dense layers -> Output (1)

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=1, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, bidirectional=True, dropout=dropout if num_layers>1 else 0.0)
        self.fc1 = nn.Linear(hidden_size*2, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out, _ = self.lstm(x)           # out: (batch, seq_len, hidden*2)
        out = out[:, -1, :]             # last time step
        out = self.dropout(self.relu(self.fc1(out)))
        out = self.fc2(out)
        return out

### 5.4 Transformer Encoder model (PyTorch)
Simple Transformer Encoder: input linear -> positional encoding -> encoder -> pooling -> MLP -> output

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return x

class TransformerModel(nn.Module):
    def __init__(self, input_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2):
        super().__init__()
        self.input_fc = nn.Linear(input_size, d_model)
        self.pos_enc = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        x = self.input_fc(x)  # (batch, seq_len, d_model)
        x = self.pos_enc(x)
        x = self.transformer_encoder(x)  # (batch, seq_len, d_model)
        x = x.permute(0,2,1)  # (batch, d_model, seq_len)
        x = self.pool(x).squeeze(2)  # (batch, d_model)
        out = self.fc(x)
        return out

### 5.5 Training helpers
- `train_one_epoch` performs a single epoch training pass.
- `validate` computes MSE loss on the test set.
- The main `train_model_torch` will print progress in the requested format:
  `Epoch i/N Train loss: <value> | Test loss: <value>`

In [ ]:
def compute_mse_on_loader(model, loader, device=DEVICE):
    model.eval()
    criterion = nn.MSELoss()
    total_loss = 0.0
    total_samples = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device); yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            total_loss += loss.item() * xb.size(0)
            total_samples += xb.size(0)
    return total_loss / total_samples if total_samples>0 else 0.0

def train_model_torch(model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, epochs+1):
        model.train()
        running_loss = 0.0
        total = 0
        for xb, yb in train_loader:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            opt.step()
            running_loss += loss.item() * xb.size(0)
            total += xb.size(0)

        train_loss = running_loss / total if total>0 else 0.0
        # compute test loss (MSE) on full test_loader
        test_loss = compute_mse_on_loader(model, test_loader, device)

        # Print progress in the required format
        print(f"Epoch {epoch}/{epochs} Train loss: {train_loss:.6f} | Test loss: {test_loss:.6f}")

        # Early stopping logic (based on test loss)
        if test_loss < best_loss - 1e-8:
            best_loss = test_loss
            best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            if best_state is not None:
                model.load_state_dict(best_state)
            print(f"Early stopping at epoch {epoch}")
            break

    # if best state exists, load it
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

### 5.6 Train LSTM and Transformer
We will train both models and print epoch progress. Training will use the `train_loader` and `test_loader` defined earlier.

In [ ]:
# Prepare DataLoaders (train and test)
train_loader = DataLoader(SeqDataset(X_train_dl, y_train_dl), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(SeqDataset(X_test_dl, y_test_dl), batch_size=BATCH_SIZE, shuffle=False)

input_size = X_train_dl.shape[2]

# LSTM
lstm_model = LSTMModel(input_size=input_size, hidden_size=64, num_layers=1, dropout=0.2)
print("Training LSTM...")
lstm_model = train_model_torch(lstm_model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE)
torch.save(lstm_model.state_dict(), 'pytorch_lstm.pt')
print("LSTM training complete and model saved as pytorch_lstm.pt")

# Transformer
transformer_model = TransformerModel(input_size=input_size, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.2)
print("Training Transformer...")
transformer_model = train_model_torch(transformer_model, train_loader, test_loader, lr=1e-3, epochs=200, patience=40, device=DEVICE)
torch.save(transformer_model.state_dict(), 'pytorch_transformer.pt')
print("Transformer training complete and model saved as pytorch_transformer.pt")

## 6) Predictions & Evaluation (DL)
- We predict on the DL test set.
- We invert the target scaler to get GDP in original units.
- Compute MAE, RMSE, R² on DL test set.

In [ ]:
# Prediction helper
def predict_torch(model, X, device=DEVICE, batch_size=64):
    model.eval()
    ds = SeqDataset(X, np.zeros((len(X),)))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    preds = []
    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            out = model(xb).cpu().numpy().reshape(-1)
            preds.append(out)
    return np.concatenate(preds, axis=0)

# LSTM predictions (scaled), then inverse transform
y_pred_lstm_scaled = predict_torch(lstm_model, X_test_dl)
y_pred_lstm_unscaled = TARGET_SCALER.inverse_transform(y_pred_lstm_scaled.reshape(-1,1)).reshape(-1)
y_test_dl_unscaled = TARGET_SCALER.inverse_transform(y_test_dl.reshape(-1,1)).reshape(-1)

metrics_lstm = evaluate_metrics(y_test_dl_unscaled, y_pred_lstm_unscaled)
print("LSTM metrics (DL test set):")
for k,v in metrics_lstm.items():
    print(f"{k}: {v:.6f}")

# Transformer predictions
y_pred_trans_scaled = predict_torch(transformer_model, X_test_dl)
y_pred_trans_unscaled = TARGET_SCALER.inverse_transform(y_pred_trans_scaled.reshape(-1,1)).reshape(-1)

metrics_trans = evaluate_metrics(y_test_dl_unscaled, y_pred_trans_unscaled)
print("\nTransformer metrics (DL test set):")
for k,v in metrics_trans.items():
    print(f"{k}: {v:.6f}")

## 7) Fair comparison: align ML predictions to the DL-test calendar years
We compute ML model predictions for the ML test period, then restrict ML test predictions to the DL test years (intersection) so all 4 models are compared on identical calendar years. We then compute MAE, RMSE, R² for each model on the intersection years and present a table and plots.

In [ ]:
# ML predictions are on X_test (years_ml)
poly_pred_full = poly_pred  # length = len(y_test)
xgb_pred_full = xgb_pred

# Find intersection of years between ML test and DL test
years_ml_list = list(years_ml)
years_dl_list = list(years_dl)
intersect_indices_in_ml = [i for i, yv in enumerate(years_ml_list) if yv in years_dl_list]
intersect_years = [years_ml_list[i] for i in intersect_indices_in_ml]

# ML values for intersection indices
ml_actual_on_intersect = y_test[intersect_indices_in_ml]
ml_poly_on_intersect = poly_pred_full[intersect_indices_in_ml]
ml_xgb_on_intersect = xgb_pred_full[intersect_indices_in_ml]

# DL mapping year -> index in DL test
dl_year_to_idx = {y: idx for idx, y in enumerate(years_dl_list)}
dl_indices_aligned = [dl_year_to_idx[y] for y in intersect_years]
dl_lstm_on_intersect = y_pred_lstm_unscaled[dl_indices_aligned]
dl_trans_on_intersect = y_pred_trans_unscaled[dl_indices_aligned]

# Evaluate each model on the same years
metrics_poly_i = evaluate_metrics(ml_actual_on_intersect, ml_poly_on_intersect)
metrics_xgb_i = evaluate_metrics(ml_actual_on_intersect, ml_xgb_on_intersect)
metrics_lstm_i = evaluate_metrics(ml_actual_on_intersect, dl_lstm_on_intersect)
metrics_trans_i = evaluate_metrics(ml_actual_on_intersect, dl_trans_on_intersect)

comparison_df = pd.DataFrame([
    {"Model":"Polynomial Regression", **metrics_poly_i},
    {"Model":"XGBoost", **metrics_xgb_i},
    {"Model":"LSTM (PyTorch)", **metrics_lstm_i},
    {"Model":"Transformer (PyTorch)", **metrics_trans_i}
])

print("\n=== Fair comparison on same calendar years ===")
print("Years compared:", intersect_years)
display(comparison_df)

# Save CSV
comparison_df.to_csv('comparison_same_years_pytorch_colab.csv', index=False)
print("Saved comparison CSV: comparison_same_years_pytorch_colab.csv")

# Plots
plt.figure(figsize=(10,6))
comparison_df_plot = comparison_df.set_index('Model')[['RMSE','MAE','R2']]
comparison_df_plot.plot(kind='bar', figsize=(10,6))
plt.title('Model comparison (same calendar years): MAE, RMSE, R2')
plt.tight_layout()
plt.savefig('comparison_metrics_bar_colab.png', dpi=150)
plt.show()

# Actual vs Predictions plot (same years)
plt.figure(figsize=(12,6))
yrs = intersect_years
plt.plot(yrs, ml_actual_on_intersect, marker='o', label='Actual')
plt.plot(yrs, ml_poly_on_intersect, marker='o', label='Polynomial')
plt.plot(yrs, ml_xgb_on_intersect, marker='o', label='XGBoost')
plt.plot(yrs, dl_lstm_on_intersect, marker='o', label='LSTM')
plt.plot(yrs, dl_trans_on_intersect, marker='o', label='Transformer')
plt.xlabel('Year'); plt.ylabel('GDP (Constant Prices)')
plt.title('Actual vs Predicted (All models) — same years')
plt.legend(); plt.tight_layout()
plt.savefig('actual_vs_pred_same_years_colab.png', dpi=150)
plt.show()

## 8) Save models & concluding notes
- Saved files: `polynomial_pipeline.joblib`, `xgboost_model.joblib`, `pytorch_lstm.pt`, `pytorch_transformer.pt`, scalers, CSV, and plots.

In [ ]:
print("Files saved in Colab working directory:")
for f in ['polynomial_pipeline.joblib','xgboost_model.joblib','pytorch_lstm.pt','pytorch_transformer.pt',
          'scaler_ml.joblib','scaler_features_dl.joblib','scaler_target_dl.joblib',
          'comparison_same_years_pytorch_colab.csv','comparison_metrics_bar_colab.png','actual_vs_pred_same_years_colab.png']:
    if os.path.exists(f):
        print(" -", f)
    else:
        print(" -", f, "(not found, maybe training stopped early before save)")